# 01 — TİTCK ve SKRS Temizliği & Birleştirilmesi (İP1)

**Sorumlu:** Kişi A  
**Rol:** Türk İlaç Verisi Temizliği ve Birleştirmesi  
**Girdi:** `data/raw/ruhsatli_2026-10-14.xlsx`, `data/raw/skrs_2026-10-14.xlsx`  
**Çıktılar:** `urun.csv`, `atc_celiski_siniflari.csv`, `barkod_tekrar.csv`, `eksik_kayit.csv`, `docs/atc_celiski_kontrol.md`

In [1]:
import sys
import re
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

# src modülünü bağla
kok = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(kok))
from src.normalize import tr_lower, barkod_temizle, gtin_gecerli

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('✓ Gerekli kütüphaneler ve src.normalize modülü başarıyla yüklendi.')


✓ Gerekli kütüphaneler ve src.normalize modülü başarıyla yüklendi.


## 1. Ham Verileri Yükleme ve TÜM METİN SÜTUNLARINI Temizleme (Adım 1)
Tüm metin sütunlarında baş/son boşluklar ve görünmez karakterler atılır.

In [2]:
r_yol = kok / 'data' / 'raw' / 'ruhsatli_2026-10-14.xlsx'
s_yol = kok / 'data' / 'raw' / 'skrs_2026-10-14.xlsx'

df_titck = pd.read_excel(r_yol, sheet_name='RUHSATLI ÜRÜNLER LİSTESİ', header=1, dtype=str)
df_skrs_aktif = pd.read_excel(s_yol, sheet_name='AKTİF ÜRÜNLER LİSTESİ', header=2, dtype=str)
df_skrs_pasif = pd.read_excel(s_yol, sheet_name='PASİF ÜRÜNLER LİSTESİ', header=2, dtype=str)

# ADIM 1: Tüm metin sütunlarında baş/son boşluk ve görünmez karakterleri at
_GORUNMEZ_RE = re.compile(r'[​‌‍⁠﻿]')

def strip_all(df):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', _GORUNMEZ_RE.sub('', x)).strip() if isinstance(x, str) else x)
        df[col] = df[col].replace('', np.nan)
    return df

df_titck = strip_all(df_titck)
df_skrs_aktif = strip_all(df_skrs_aktif)
df_skrs_pasif = strip_all(df_skrs_pasif)

print(f'TİTCK Ruhsatlı: {df_titck.shape[0]:,d} satır | SKRS Aktif: {df_skrs_aktif.shape[0]:,d} | SKRS Pasif: {df_skrs_pasif.shape[0]:,d}')

TİTCK Ruhsatlı: 23,105 satır | SKRS Aktif: 7,949 | SKRS Pasif: 10,212


## 2. Barkod Kuralları ve TİTCK Tekrar Analizi (Adım 2 - Kural 2 & 3)
TİTCK mükerrerlerinde askıda olmayan ve tarihi en yeni olan tutulur, elenenler `barkod_tekrar.csv`ye kaydedilir.

In [3]:
aski_col = [c for c in df_titck.columns if 'ASKIDA' in str(c)][0]
df_titck['ASKIDA_DURUMU'] = df_titck[aski_col].fillna('0')

df_titck['barkod'] = df_titck['BARKOD'].apply(barkod_temizle)
df_skrs_aktif['barkod'] = df_skrs_aktif['Barkod'].apply(barkod_temizle)
df_skrs_pasif['barkod'] = df_skrs_pasif['Barkod'].apply(barkod_temizle)

# Kural 3: TİTCK Mükerrer Yönetimi (Askıda olmayan ve tarihi en yeni olan tutulur)
df_titck['tarih_dt'] = pd.to_datetime(df_titck['RUHSAT TARİHİ'], errors='coerce')
df_titck['aski_prio'] = (df_titck['ASKIDA_DURUMU'] == '0').astype(int)

titck_sirali = df_titck.sort_values(by=['aski_prio', 'tarih_dt'], ascending=[False, False])

mask_notna = titck_sirali['barkod'].notna()
titck_tekil_dolu = titck_sirali[mask_notna].drop_duplicates(subset=['barkod'], keep='first')
titck_tekrar = titck_sirali[mask_notna][titck_sirali[mask_notna].duplicated(subset=['barkod'], keep='first')].copy()
titck_bos = titck_sirali[~mask_notna]
titck_tekil = pd.concat([titck_tekil_dolu, titck_bos], ignore_index=True)

# Elenenleri kaydet
tekrar_cols = ['BARKOD', 'ÜRÜN ADI', 'ETKİN MADDE', 'ATC KODU', 'RUHSAT SAHİBİ', 'RUHSAT TARİHİ', 'RUHSAT NUMARASI', 'ASKIDA_DURUMU']
titck_tekrar_clean = titck_tekrar[tekrar_cols].rename(columns={
    'BARKOD': 'barkod', 'ÜRÜN ADI': 'urun_adi', 'ETKİN MADDE': 'etkin_madde',
    'ATC KODU': 'atc_kodu', 'RUHSAT SAHİBİ': 'ruhsat_sahibi', 'RUHSAT TARİHİ': 'ruhsat_tarihi',
    'RUHSAT NUMARASI': 'ruhsat_no', 'ASKIDA_DURUMU': 'askida_durumu'
})
titck_tekrar_clean.to_csv(kok / 'data' / 'interim' / 'barkod_tekrar.csv', index=False, encoding='utf-8')
print(f'TİTCK Tekil: {len(titck_tekil):,d} | Elenen Mükerrer: {len(titck_tekrar_clean):,d} (-> data/interim/barkod_tekrar.csv)')

TİTCK Tekil: 23,053 | Elenen Mükerrer: 52 (-> data/interim/barkod_tekrar.csv)


## 3. SKRS Tekilleştirme (Adım 3)
Aynı barkod hem Aktif hem Pasif sekmede varsa Aktif satır alınır.

In [4]:
df_skrs_aktif['durum'] = 'Aktif'
df_skrs_pasif['durum'] = 'Pasif'

skrs_hepsi = pd.concat([df_skrs_aktif, df_skrs_pasif], ignore_index=True)
skrs_hepsi['aktif_prio'] = (skrs_hepsi['durum'] == 'Aktif').astype(int)
skrs_sirali = skrs_hepsi.sort_values(by=['aktif_prio'], ascending=[False])
skrs_tekil = skrs_sirali.drop_duplicates(subset=['barkod'], keep='first').copy()
print(f'SKRS Tekil Satır Sayısı: {len(skrs_tekil):,d}')

SKRS Tekil Satır Sayısı: 18,161


## 4. Left Join ve Assert (Adım 4)
Ruhsatlı listeye SKRS barkodla soldan birleştirilir. Satır sayısı artmadığı assert ile doğrulanır.

In [5]:
skrs_sub = skrs_tekil[['barkod', 'ATC Kodu', 'ATC Adı', 'durum', 'İlaç Adı']].rename(columns={
    'ATC Kodu': 'atc_skrs_raw', 'ATC Adı': 'atc_adi_skrs', 'durum': 'durum_skrs'
})

merged = pd.merge(titck_tekil, skrs_sub, on='barkod', how='left')

# ADIM 4 ASSERT KURALI: Birleştirmeden sonra ruhsatlı satır sayısı artmamalı!
assert len(merged) == len(titck_tekil), f'KRİTİK HATA: Left join sonrası satır sayısı arttı! {len(titck_tekil)} -> {len(merged)}'

merged['kaynak'] = 'Ruhsatlı'
merged['urun_adi'] = merged['ÜRÜN ADI']
merged['etkin_madde_ham'] = merged['ETKİN MADDE']
merged['atc_ruhsat_raw'] = merged['ATC KODU']
merged['durum'] = merged['durum_skrs'].where(merged['durum_skrs'].notna(), merged['ASKIDA_DURUMU'].map({'0': 'Aktif'}).fillna('Pasif'))
print(f'✓ Left join ve assert doğrulandı. Ruhsatlı ürün sayısı: {len(merged):,d}')

✓ Left join ve assert doğrulandı. Ruhsatlı ürün sayısı: 23,053


## 5. Kural 5: SKRS'ten Ek Ürünler (Adım 5)
SKRS'te aktif olup ruhsatlıda olmayanlar eklenir (kaynak='SKRS').

In [6]:
titck_b_set = set(titck_tekil['barkod'].dropna())
skrs_eklenecek = skrs_tekil[(skrs_tekil['durum'] == 'Aktif') & (~skrs_tekil['barkod'].isin(titck_b_set))].copy()

ek_df = pd.DataFrame({
    'barkod': skrs_eklenecek['barkod'],
    'urun_adi': skrs_eklenecek['İlaç Adı'],
    'etkin_madde_ham': skrs_eklenecek['ATC Adı'],
    'atc_ruhsat_raw': np.nan,
    'atc_skrs_raw': skrs_eklenecek['ATC Kodu'],
    'atc_adi_skrs': skrs_eklenecek['ATC Adı'],
    'durum': 'Aktif',
    'kaynak': 'SKRS'
})

urun_full = pd.concat([merged, ek_df], ignore_index=True)
print(f'Eklenen Aktif SKRS Ürün: {len(ek_df):,d} | Toplam Birleştirilmiş: {len(urun_full):,d}')

Eklenen Aktif SKRS Ürün: 851 | Toplam Birleştirilmiş: 23,904


## 6. Kural 6: Boş Kayıtlar ve Analizden Çıkarma (Adım 6)
Etkin maddesi boş olan ruhsatlı ilaçlar SKRS ATC Adı ile tamamlanır. Tamamlanamayanlar `eksik_kayit.csv`ye yazılır ve analizden çıkarılır.

In [7]:
# Etkin maddesi boş olanları SKRS ATC Adı ile tamamla
kural6_mask = urun_full['etkin_madde_ham'].isna() & urun_full['atc_adi_skrs'].notna()
urun_full.loc[kural6_mask, 'etkin_madde_ham'] = urun_full.loc[kural6_mask, 'atc_adi_skrs']

# Hala boş olanlar (Etkin maddesi yok VEYA Her İki ATC'si birden yok)
c_etkin_bos = urun_full['etkin_madde_ham'].isna()
c_atc_ikisi_bos = urun_full['atc_ruhsat_raw'].isna() & urun_full['atc_skrs_raw'].isna()

eksik_df = urun_full[c_etkin_bos | c_atc_ikisi_bos].copy()

# eksik_df üzerinde eksik_turu belirleme
m_etkin = eksik_df['etkin_madde_ham'].isna()
m_atc = eksik_df['atc_ruhsat_raw'].isna() & eksik_df['atc_skrs_raw'].isna()
eksik_df['eksik_turu'] = np.where(m_etkin & m_atc, 'etkin_madde_ve_atc_yok',
                         np.where(m_etkin, 'etkin_madde_yok', 'atc_yok'))

eksik_cols = ['barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat_raw', 'atc_skrs_raw', 'atc_adi_skrs', 'durum', 'kaynak', 'eksik_turu']
eksik_df[eksik_cols].to_csv(kok / 'data' / 'interim' / 'eksik_kayit.csv', index=False, encoding='utf-8')

# ADIM 6 ZORUNLULUĞU: Tamamlanamayanları analizden ÇIKAR!
urun_full = urun_full[~(c_etkin_bos | c_atc_ikisi_bos)].reset_index(drop=True)
print(f'Kurtarılan Kayıt (LOXIBIN): {int(kural6_mask.sum())} | Analizden Çıkarılan Eksik: {len(eksik_df):,d} | Kalan: {len(urun_full):,d}')
# Kural 6 İzlenebilirlik: Etkin maddesi veya ATC'si SKRS'ten tamamlanan kayıtları işaretle
c_atc_tamamlandi = (urun_full['kaynak'] == 'Ruhsatlı') & urun_full['atc_ruhsat_raw'].isna() & urun_full['atc_skrs_raw'].notna()
urun_full['kural_6_skrs_tamamlandi'] = (kural6_mask | c_atc_tamamlandi).astype(int)
print(f'Kural 6 SKRS Tamamlandı İşaretlenen Satır Sayısı: {urun_full["kural_6_skrs_tamamlandi"].sum()}')


Kurtarılan Kayıt (LOXIBIN): 1 | Analizden Çıkarılan Eksik: 103 | Kalan: 23,801
Kural 6 SKRS Tamamlandı İşaretlenen Satır Sayısı: 14


## 7. ATC Temizliği, Kural 4, Kural 2 (Adım 7 & 8)
ATC kodları standartlaştırılır. Geçersiz GTIN barkodları `barkod_gecersiz=1` ile işaretlenir.

In [8]:
def clean_atc(x):
    if pd.isna(x): return None
    s = tr_lower(x)
    return s.upper() if s else None

urun_full['atc_ruhsat'] = urun_full['atc_ruhsat_raw'].apply(clean_atc)
urun_full['atc_skrs'] = urun_full['atc_skrs_raw'].apply(clean_atc)

# Kural 2: Geçersiz barkodları işaretle (Silme)
urun_full['barkod_gecersiz'] = (~urun_full['barkod'].apply(gtin_gecerli)).astype(int)
print(f'Geçersiz GTIN Barkod Sayısı: {urun_full["barkod_gecersiz"].sum():,d}')
# TİTCK Resmi Veri Hatası Düzeltmesi (docs/acik_sorular.md Madde 13 uyarınca)
# TEKNAZOL 200 MG TABLET (Barkod: 8699540014800, 8699540014817, 8699540014824) ATC kodu
# sehven girilmiş M01AE03 (ketoprofen) yerine resmi J02AB02 (ketoconazole) atanır.
teknazol_barkodlar = {'8699540014800', '8699540014817', '8699540014824', 8699540014800, 8699540014817, 8699540014824}
teknazol_mask = urun_full['barkod'].isin(teknazol_barkodlar)
urun_full.loc[teknazol_mask, 'atc_ruhsat'] = 'J02AB02'
urun_full['atc_kaynak_hatasi_duzeltildi'] = teknazol_mask.astype(int)
print(f'TEKNAZOL (ketokonazol) ATC düzeltildi (J02AB02): {teknazol_mask.sum()} ürün')


Geçersiz GTIN Barkod Sayısı: 184
TEKNAZOL (ketokonazol) ATC düzeltildi (J02AB02): 3 ürün


## 8. ATC Çelişki Sınıfları ve Kural 8 (Adım 8 & 10)
ATC kodları 5 sınıfa ayrılır. Kural 8 gereği `atc_final` ve `atc_celiski` atanır. `atc_celiski_siniflari.csv` kaydedilir.

In [9]:
def celiski_sinifi(row):
    r, s = row['atc_ruhsat'], row['atc_skrs']
    if pd.isna(r) or pd.isna(s): return 'bos'
    if r == s: return 'bosluk'
    if r.startswith(s) or s.startswith(r): return 'duzey_farki'
    if len(r) >= 5 and len(s) >= 5 and r[:5] == s[:5]: return 'ayni_grup_farkli_madde'
    return 'farkli_grup'

urun_full['atc_celiski_sinifi'] = urun_full.apply(celiski_sinifi, axis=1)

def atc_final_belirle(row):
    r, s, sinif = row['atc_ruhsat'], row['atc_skrs'], row['atc_celiski_sinifi']
    if sinif == 'bosluk': return r, 0
    elif sinif == 'duzey_farki': return (r if len(r) >= len(s) else s), 0
    elif sinif in ('ayni_grup_farkli_madde', 'farkli_grup'): return r, 1
    elif sinif == 'bos': return (r if pd.notna(r) else s), 0
    return r, 0

finals = urun_full.apply(atc_final_belirle, axis=1)
urun_full['atc_final'] = [f[0] for f in finals]
urun_full['atc_celiski'] = [f[1] for f in finals]

# Kural 4: V06/V07 kapsam dışı işaretle (atc_final oluştuktan sonra)
urun_full['kapsam_disi'] = urun_full['atc_final'].fillna('').str.startswith(('V06', 'V07')).astype(int)

# Çelişki CSV kaydet
celiski_df = urun_full[urun_full['atc_celiski_sinifi'] != 'bosluk'][[
    'barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs', 'atc_adi_skrs', 'atc_celiski_sinifi'
]]
celiski_df.to_csv(kok / 'data' / 'interim' / 'atc_celiski_siniflari.csv', index=False, encoding='utf-8')
print(f'atc_celiski_siniflari.csv kaydedildi ({len(celiski_df):,d} satır).')

atc_celiski_siniflari.csv kaydedildi (12,579 satır).


## 9. Elle Kontrol - Kişi C İçin Görüntüleme (Adım 9)
`farkli_grup` sınıfından 20 ürünün incelenmesi. Detaylı analiz `docs/atc_celiski_kontrol.md` dosyasındadır.

In [10]:
print('--- KİŞİ C İÇİN ATC ÇELİŞKİ KONTROLÜ (farkli_grup) ---')
print('Detaylı Rapor: docs/atc_celiski_kontrol.md dosyasındadır.')
ornek_20 = celiski_df[celiski_df['atc_celiski_sinifi'] == 'farkli_grup'].head(20)
display(ornek_20[['barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs']])

--- KİŞİ C İÇİN ATC ÇELİŞKİ KONTROLÜ (farkli_grup) ---
Detaylı Rapor: docs/atc_celiski_kontrol.md dosyasındadır.


,barkod,urun_adi,etkin_madde_ham,atc_ruhsat,atc_skrs
344,8685073367001,"FLAGYL 500 MG FİLM TABLET, 20 ADET",metronidazol,P01AB01,J01XD01
528,8699548095603,Femoston 1 mg/10 mg Film Kaplı Tablet (28 Adet),"Estradiol hemihidrat, Didrogesteron",G03FA14,G03FB08
530,8699548095610,Femoston 2 mg/10 mg Film Kaplı Tablet (28 Adet),"Estradiol hemihidrat, Didrogesteron",G03FA14,G03FB08
564,8699074090509,Admiraz 1 mg Film Kaplı Tablet (28 adet),Siponimod fumarik asit,L04AE03,C09DX04
708,8699717150010,"ATOMINEX 10 MG KAPSÜL, 28 KAPSÜL",amoksetin hidroklorür,N06BA09,J01CA04
1052,8680150570268,"PEDITUS SURUP, 100 ML",other cold preparations,R05X,N02BE51
1157,8681801091378,"EMEDUR 200 MG FILM KAPLI TABLET, 20 ADET",trimetobenzamid hcl,A04AD,R06AA10
1196,8699624770028,"Rystiggo 280 mg/2 ml Enjeksiyonluk Çözelti, 1 ...",Rozanolixizumab,L04AG16,L04AL02
1345,8680150350112,"Nimejel Plus %1+ %5 Jel, 50 G","Nimesulid, Lidokain",M02AA26,D10AF05
1444,8699578040079,"Bio-sidoz 1000 mg Gastro Rezistan Tablet, 50 ADET",Sodyum bikarbonat,A02AX,A02AH


## 10. Nihai Kayıt ve Özet Tablo (Adım 10)
Nihai `data/interim/urun.csv` dosyası kaydedilir ve özet istatistik tablosu görüntülenir.

In [11]:
final_cols = [
    'barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs',
    'atc_adi_skrs', 'durum', 'kaynak', 'atc_final', 'atc_celiski',
    'kapsam_disi', 'barkod_gecersiz', 'kural_6_skrs_tamamlandi',
    'atc_kaynak_hatasi_duzeltildi',
]
urun_df = urun_full[final_cols].copy()
cikis_yol = kok / 'data' / 'interim' / 'urun.csv'
urun_df.to_csv(cikis_yol, index=False, encoding='utf-8')

print(f'\n✓ {cikis_yol} başarıyla kaydedildi! Toplam Satır: {len(urun_df):,d}')

ozet_tablo = pd.DataFrame([
    {'Metrik': 'TİTCK Başlangıç Satır Sayısı', 'Değer': len(df_titck)},
    {'Metrik': 'Mükerrer Barkod Nedeniyle Elenen (TİTCK)', 'Değer': len(titck_tekrar_clean)},
    {'Metrik': "SKRS'ten Eklenen Aktif Ürünler (Kural 5)", 'Değer': len(ek_df)},
    {'Metrik': 'Kural 6 ile Kurtarılan Kayıt', 'Değer': int(kural6_mask.sum())},
    {'Metrik': 'TİTCK Kaynak Hatası Düzeltilen (TEKNAZOL)', 'Değer': int(urun_full['atc_kaynak_hatasi_duzeltildi'].sum())},
    {'Metrik': 'Eksik Kayıt (Analizden Çıkarılan)', 'Değer': len(eksik_df)},
    {'Metrik': 'Kapsam Dışı İşaretlenen (V06/V07)', 'Değer': int(urun_full['kapsam_disi'].sum())},
    {'Metrik': 'Nihai Ürün Sayısı (urun.csv)', 'Değer': len(urun_df)},
    {'Metrik': 'ATC Çelişkisi: bosluk (birebir aynı)', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'bosluk').sum())},
    {'Metrik': 'ATC Çelişkisi: duzey_farki', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'duzey_farki').sum())},
    {'Metrik': 'ATC Çelişkisi: ayni_grup_farkli_madde', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'ayni_grup_farkli_madde').sum())},
    {'Metrik': 'ATC Çelişkisi: farkli_grup', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'farkli_grup').sum())},
    {'Metrik': 'ATC Çelişkisi: bos (bir tarafta yok)', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'bos').sum())},
])
display(ozet_tablo)


✓ /Users/mervesudeyanar/Desktop/ddi-final/data/interim/urun.csv başarıyla kaydedildi! Toplam Satır: 23,801


,Metrik,Değer
0,TİTCK Başlangıç Satır Sayısı,23105
1,Mükerrer Barkod Nedeniyle Elenen (TİTCK),52
2,SKRS'ten Eklenen Aktif Ürünler (Kural 5),851
3,Kural 6 ile Kurtarılan Kayıt,1
4,TİTCK Kaynak Hatası Düzeltilen (TEKNAZOL),3
5,Eksik Kayıt (Analizden Çıkarılan),103
6,Kapsam Dışı İşaretlenen (V06/V07),456
7,Nihai Ürün Sayısı (urun.csv),23801
8,ATC Çelişkisi: bosluk (birebir aynı),11222
9,ATC Çelişkisi: duzey_farki,49
